# 01 · Dados: SIVEP-Gripe (SRAG) → painel semanal de VSR em menores de 2 anos

Camada de dados do projeto *Previsão probabilística do início e do pico da temporada de VSR nos estados brasileiros*.
Este notebook **só** baixa, valida, harmoniza e salva. Exploração e modelos leem exclusivamente `data/processed/`.

**Saídas** (contrato com os notebooks seguintes; esquema em `data/processed/metadata.json`)

| Arquivo | Grão | Conteúdo |
|---|---|---|
| `interim/casos_menores2.parquet` | caso | base limpa, uma linha por notificação de SRAG hospitalizada em < 2 anos |
| `processed/painel_semanal.parquet` | criterio_uf × unidade × faixa_etaria × semana | contagens observadas, testagem, positividade, limites do VSR corrigido pela testagem, séries imputadas no período 2020–21, flags; termina na última semana completa |
| `processed/triangulo_atraso.parquet` | criterio_uf × UF × faixa × semana × atraso | casos por semana de sintomas e atraso de disponibilidade: reconstrói qualquer versão em tempo real |
| `processed/completude_atraso.parquet` | serie × região × atraso | fração acumulada disponível $F_r(d)$ |
| `processed/climatologia.parquet` | unidade × faixa × serie × temporada_alvo × semana_epi | perfil sazonal só com temporadas anteriores à alvo (sem vazamento) e versão completa (EDA) |
| `processed/hierarquia_unidades.csv`, `matriz_S.csv` | | hierarquia Brasil → região → UF e matriz de agregação $S$ |
| `processed/tiers_uf.csv` | UF | classificação A/B/C por volume de casos (regra fixada *a priori*) |
| `processed/qc_*.csv`, `raw/manifesto_download.json` | | auditoria: filtros, cobertura por ano, colunas por arquivo, hashes |

**Definições principais**

- Caso: notificação de SRAG com `HOSPITAL = 1`, idade < 24 meses, semana de primeiros sintomas $t$ (semana epidemiológica, domingo a sábado).
- VSR confirmado: `PCR_VSR = 1` ou `AN_VSR = 1`.
- Disponibilidade: registro em $a^{\text{reg}} = $ `DT_DIGITA`; VSR em $a^{\text{vsr}} = \max(\texttt{DT\_DIGITA}, \texttt{DT\_PCR}, \texttt{DT\_RES\_AN})$ (datas de resultado só quando o respectivo teste foi positivo para VSR).
- **Série-alvo principal: `vsr` (confirmado).** `srag` é o alvo de sensibilidade.
- `testado_painel`: VSR positivo, ou `POS_PCROUT` ∈ {1,2}, ou `POS_AN_OUT` ∈ {1,2}. Nos dados de 2019–2026 esses campos (e `POS_PCRFLU`, `PCR_FLUASU`, `PCR_VSR`) só são preenchidos quando o teste detectou algum agente: `testado_painel` ≈ *teste com agente detectado*, não "painel realizado".
- `lab_resultado`: `PCR_RESUL` ou `RES_AN` ∈ {1,2} (detectável ou não detectável, qualquer agente, incluindo testes só para SARS-CoV-2), ou VSR positivo.
- VSR corrigido pela testagem, por UF $u$ e faixa $f$, com denominador $n \in \{\texttt{lab\_resultado}, \texttt{testado\_painel}\}$:
$$\tilde p_{u,t} = \frac{\sum_{k=0}^{w-1} y_{u,t-k} + \kappa\,\bar p_{r(u),t}}{\sum_{k=0}^{w-1} n_{u,t-k} + \kappa},\qquad \hat y_{u,t} = N_{u,t}\,\tilde p_{u,t}$$
com $y$ = VSR positivos, $N$ = SRAG total, $\bar p_{r(u),t}$ = positividade da região na mesma janela (só passado: janela *trailing*). `vsr_est_inf` ($n$ = `lab_resultado`; negativos de testes só para SARS-CoV-2 deprimem $\tilde p$) e `vsr_est_sup` ($n$ = `testado_painel`; exclui todos os negativos) são **limites**, não estimativas pontuais. Região e Brasil são somas das UFs ⇒ hierarquia coerente.
- O painel termina na última semana epidemiológica completa (sábado ≤ data de corte).
- Período 2020–2021 (`quebra_npi`): mantido observado e acompanhado de duas séries imputadas por climatologia (`*_imput_passado`, sem vazamento; `*_imput_ref`, com todas as temporadas de referência, só para EDA/sensibilidade).

**Limitações conhecidas**: nomes e códigos seguem o dicionário do SIVEP-Gripe e devem ser confirmados no dicionário publicado junto aos arquivos (a célula de QC lista o que foi encontrado por ano). `DT_DIGITA` não registra edições posteriores do registro; a data de resultado reduz, mas não elimina, o viés otimista da reconstrução em tempo real.

## 0. Configuração

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import sys
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests

from vsr_comum import verificar_ambiente

verificar_ambiente()   # versões fixadas em environment.yml
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("vsr.dados")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

In [2]:
@dataclass
class Config:
    # Diretórios
    raiz: Path = Path("data")
    # Arquivos-fonte
    anos: list[int] = field(default_factory=lambda: list(range(2019, 2027)))
    anos_obrigatorios: list[int] = field(default_factory=lambda: list(range(2019, 2027)))
    formatos: tuple[str, ...] = ("parquet", "csv")      # ordem de preferência
    arquivos_manuais: dict[int, str] = field(default_factory=dict)   # {ano: url}
    arquivos_locais: dict[int, str] = field(default_factory=dict)    # {ano: caminho}
    offline: bool = os.environ.get("VSR_OFFLINE") == "1"  # True: não acessa a rede (reprocessa o cache)
    forcar_download: bool = False
    tentativas_download: int = 6
    timeout_conexao_s: int = 30
    timeout_leitura_s: int = 300
    # Definição de caso
    idade_max_meses: int = 24
    somente_hospitalizados: bool = True
    incluir_hospital_ausente: bool = False     # sensibilidade: mantém HOSPITAL vazio (coluna hospital_ausente)
    criterios_uf: tuple[str, ...] = ("residencia", "notificacao")   # o primeiro é o principal
    faixas_meses: tuple[int, ...] = (0, 6, 12, 24)                  # [0,6), [6,12), [12,24)
    # Calendário e períodos
    inicio_painel: str = "2018-12-30"          # domingo da SE 1/2019
    quebra_ini: str = "2020-03-01"             # início das NPIs (SE 10/2020)
    quebra_fim: str = "2021-12-31"
    imunizacao_ini: str = "2026-02-01"         # nirsevimabe (nota técnica do MS)
    vacina_materna_ini: str | None = None      # preencher com a data da nota técnica, se usada
    semana_inicio_temporada: dict[str, int] = field(default_factory=dict)  # {região: SE}; vazio = SE 1
    temporadas_excluidas_climatologia: tuple[int, ...] = (2020, 2021)
    # Positividade corrigida
    janela_positividade: int = 3               # semanas (trailing)
    kappa_shrinkage: float = 5.0               # pseudo-contagem em direção à região
    # Atraso
    atraso_max_semanas: int = 26               # último bucket = >= atraso_max
    completude_flag: float = 0.95
    temporadas_completude: int = 3             # nº de temporadas recentes para F_r(d)
    # Tiers de UF (regra a priori, sobre VSR confirmado por residência)
    tier_temporadas: tuple[int, ...] = (2023, 2024, 2025)
    tier_a_min: int = 300
    tier_b_min: int = 100
    # Temporada usável (climatologia e tiers) se ≥ esta fração das semanas tiver cobertura completa
    cobertura_min_temporada: float = 0.9
    # Referência populacional opcional (uf, ano, pop_menores2); gerada por baixar_sinasc.py
    arquivo_populacao: str | None = "data/external/populacao.csv"
    # Colunas extras mantidas por regex (imunização etc.)
    regex_colunas_extras: str = r"NIRSE|PALIV|VSR|ANTICORP|VAC.*GEST|GESTA"

    @property
    def dir_raw(self) -> Path: return self.raiz / "raw" / "SRAG"
    @property
    def dir_interim(self) -> Path: return self.raiz / "interim"
    @property
    def dir_processed(self) -> Path: return self.raiz / "processed"


def carregar_config() -> Config:
    """Config padrão, sobrescrita opcionalmente por JSON em $VSR_DATA_CONFIG (execução headless)."""
    cfg = Config()
    caminho = os.environ.get("VSR_DATA_CONFIG")
    if caminho:
        over = json.loads(Path(caminho).read_text(encoding="utf-8"))
        for k, v in over.items():
            if not hasattr(cfg, k):
                raise KeyError(f"parâmetro desconhecido em {caminho}: {k}")
            atual = getattr(cfg, k)
            if isinstance(atual, Path):
                v = Path(v)
            elif isinstance(atual, dict):
                v = {int(a) if str(a).isdigit() else a: b for a, b in v.items()}
            elif isinstance(atual, tuple):
                v = tuple(v)
            setattr(cfg, k, v)
    for d in (cfg.dir_raw, cfg.dir_interim, cfg.dir_processed):
        d.mkdir(parents=True, exist_ok=True)
    return cfg


CFG = carregar_config()
CFG

Config(raiz=WindowsPath('data'), anos=[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], anos_obrigatorios=[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], formatos=('parquet', 'csv'), arquivos_manuais={}, arquivos_locais={}, offline=True, forcar_download=False, tentativas_download=6, timeout_conexao_s=30, timeout_leitura_s=300, idade_max_meses=24, somente_hospitalizados=True, incluir_hospital_ausente=False, criterios_uf=('residencia', 'notificacao'), faixas_meses=(0, 6, 12, 24), inicio_painel='2018-12-30', quebra_ini='2020-03-01', quebra_fim='2021-12-31', imunizacao_ini='2026-02-01', vacina_materna_ini=None, semana_inicio_temporada={}, temporadas_excluidas_climatologia=(2020, 2021), janela_positividade=3, kappa_shrinkage=5.0, atraso_max_semanas=26, completude_flag=0.95, temporadas_completude=3, tier_temporadas=(2023, 2024, 2025), tier_a_min=300, tier_b_min=100, cobertura_min_temporada=0.9, arquivo_populacao='data/external/populacao.csv', regex_colunas_extras='NIRSE|PALIV|VSR|ANTICORP|

In [3]:
# --- Geografia ---------------------------------------------------------------
REGIAO_UF = {
    "AC": "NORTE", "AP": "NORTE", "AM": "NORTE", "PA": "NORTE", "RO": "NORTE", "RR": "NORTE", "TO": "NORTE",
    "AL": "NORDESTE", "BA": "NORDESTE", "CE": "NORDESTE", "MA": "NORDESTE", "PB": "NORDESTE",
    "PE": "NORDESTE", "PI": "NORDESTE", "RN": "NORDESTE", "SE": "NORDESTE",
    "DF": "CENTRO_OESTE", "GO": "CENTRO_OESTE", "MT": "CENTRO_OESTE", "MS": "CENTRO_OESTE",
    "ES": "SUDESTE", "MG": "SUDESTE", "RJ": "SUDESTE", "SP": "SUDESTE",
    "PR": "SUL", "RS": "SUL", "SC": "SUL",
}
UFS = sorted(REGIAO_UF)
REGIOES = ["NORTE", "NORDESTE", "CENTRO_OESTE", "SUDESTE", "SUL"]
# Códigos IBGE (2 primeiros dígitos do município) → UF, para recuperar UF ausente
COD_IBGE_UF = {11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO", 21: "MA", 22: "PI",
               23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL", 28: "SE", 29: "BA", 31: "MG", 32: "ES",
               33: "RJ", 35: "SP", 41: "PR", 42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF"}

# --- Colunas do SIVEP-Gripe: nome canônico → candidatos (CONFIRMAR NO DICIONÁRIO) ----
COLUNAS = {
    "DT_SIN_PRI": ["DT_SIN_PRI"],
    "SEM_PRI":    ["SEM_PRI"],
    "DT_NOTIFIC": ["DT_NOTIFIC"],
    "DT_DIGITA":  ["DT_DIGITA"],
    "DT_PCR":     ["DT_PCR"],
    "DT_RES_AN":  ["DT_RES_AN"],
    "DT_ENCERRA": ["DT_ENCERRA"],
    "SG_UF":      ["SG_UF", "SG_UF_RES"],          # residência
    "SG_UF_NOT":  ["SG_UF_NOT"],                   # notificação
    "CO_MUN_RES": ["CO_MUN_RES"],
    "CO_MUN_NOT": ["CO_MUN_NOT"],
    "NU_IDADE_N": ["NU_IDADE_N"],
    "TP_IDADE":   ["TP_IDADE"],
    "HOSPITAL":   ["HOSPITAL"],
    "UTI":        ["UTI"],
    "EVOLUCAO":   ["EVOLUCAO"],
    "CLASSI_FIN": ["CLASSI_FIN"],
    "PCR_RESUL":  ["PCR_RESUL"],
    "RES_AN":     ["RES_AN"],
    "POS_PCROUT": ["POS_PCROUT"],
    "POS_AN_OUT": ["POS_AN_OUT"],
    "POS_PCRFLU": ["POS_PCRFLU"],
    "POS_AN_FLU": ["POS_AN_FLU"],
    "PCR_VSR":    ["PCR_VSR"],
    "AN_VSR":     ["AN_VSR"],
    "PCR_SARS2":  ["PCR_SARS2"],
    "NU_NOTIFIC": ["NU_NOTIFIC"],
    "CO_UNI_NOT": ["CO_UNI_NOT", "ID_UNIDADE"],
}
OBRIGATORIAS = ["DT_SIN_PRI", "DT_DIGITA", "NU_IDADE_N", "TP_IDADE", "HOSPITAL", "SG_UF_NOT"]
COLUNAS_DATA = ["DT_SIN_PRI", "DT_NOTIFIC", "DT_DIGITA", "DT_PCR", "DT_RES_AN", "DT_ENCERRA"]
CHAVE_DEDUP = ["NU_NOTIFIC", "DT_NOTIFIC", "SG_UF_NOT", "CO_UNI_NOT", "DT_SIN_PRI"]

## 1. Download

Descoberta dos links (o nome do arquivo traz a data de publicação e muda a cada republicação) por três rotas: API CKAN, listagem do S3, páginas do portal. Download com retomada (`Range`), novas tentativas com *backoff*, verificação de tamanho, validação do Parquet e cache por hash: arquivos já baixados e íntegros não são baixados de novo. Se um ano obrigatório faltar, o notebook **para**.

In [4]:
PORTAL = "https://dadosabertos.saude.gov.br"
S3 = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br"
SLUG = "srag-2019-a-2026"
HDR = {"User-Agent": "Mozilla/5.0 (vsr-pipeline)"}
RE_ARQ = re.compile(r"INFLUD(\d{2})-(\d{2})-(\d{2})-(\d{4})\.(parquet|csv)", re.I)

# Fallback: links vistos no portal (o do ano corrente muda; a descoberta tem prioridade)
URLS_CONHECIDAS = {
    2019: f"{S3}/SRAG/2019/INFLUD19-23-03-2026.parquet",
    2020: f"{S3}/SRAG/2020/INFLUD20-23-03-2026.parquet",
    2021: f"{S3}/SRAG/2021/INFLUD21-23-03-2026.parquet",
    2022: f"{S3}/SRAG/2022/INFLUD22-23-03-2026.parquet",
    2023: f"{S3}/SRAG/2023/INFLUD23-23-03-2026.parquet",
    2024: f"{S3}/SRAG/2024/INFLUD24-23-03-2026.parquet",
    2025: f"{S3}/SRAG/2025/INFLUD25-28-09-2026.parquet",
    2026: f"{S3}/SRAG/2026/INFLUD26-14-09-2026.parquet",
}


def data_publicacao(nome: str) -> pd.Timestamp | None:
    m = RE_ARQ.search(nome)
    return pd.Timestamp(int(m.group(4)), int(m.group(3)), int(m.group(2))) if m else None


def _extrai_urls(texto: str, formato: str) -> dict[int, str]:
    texto = texto.replace("\\/", "/")
    pat = re.compile(r"https://s3\.sa-east-1\.amazonaws\.com/ckan\.saude\.gov\.br/SRAG/(\d{4})/"
                     r"INFLUD\d{2}-\d{2}-\d{2}-\d{4}\." + formato, re.I)
    melhor: dict[int, str] = {}
    for m in pat.finditer(texto):
        ano, url = int(m.group(1)), m.group(0)
        if ano not in melhor or data_publicacao(url) > data_publicacao(melhor[ano]):
            melhor[ano] = url
    return melhor


def _via_api() -> str:
    r = requests.get(f"{PORTAL}/api/3/action/package_show", params={"id": SLUG}, headers=HDR, timeout=60)
    return json.dumps(r.json())


def _via_s3(anos) -> str:
    partes = []
    for ano in anos:
        r = requests.get(S3, params={"list-type": "2", "prefix": f"SRAG/{ano}/"}, headers=HDR, timeout=60)
        if "<ListBucketResult" not in r.text:
            raise RuntimeError("listagem do bucket não é pública")
        partes += [f"{S3}/{k}" for k in re.findall(r"<Key>([^<]+)</Key>", r.text)]
    return " ".join(partes)


def _via_paginas() -> str:
    r = requests.get(f"{PORTAL}/dataset/{SLUG}", headers=HDR, timeout=60)
    ids = list(dict.fromkeys(re.findall(rf"/dataset/{SLUG}/resource/([0-9a-f\-]{{36}})", r.text)))
    if not ids:
        raise RuntimeError("página sem links de recursos")
    return " ".join(requests.get(f"{PORTAL}/dataset/{SLUG}/resource/{i}", headers=HDR, timeout=60).text
                    for i in ids)


def descobrir_urls(anos: list[int], formato: str) -> dict[int, str]:
    achados: dict[int, str] = {}
    for nome, fn in (("API CKAN", _via_api), ("listagem S3", lambda: _via_s3(anos)), ("páginas", _via_paginas)):
        try:
            novos = _extrai_urls(fn() or "", formato)
            log.info("descoberta via %s: %d ano(s) em %s", nome, len(novos), formato)
            for a, u in novos.items():
                if a not in achados or data_publicacao(u) > data_publicacao(achados[a]):
                    achados[a] = u
            if all(a in achados for a in anos):
                break
        except Exception as e:  # noqa: BLE001 - rotas são best-effort
            log.warning("descoberta via %s falhou: %s: %s", nome, type(e).__name__, str(e)[:120])
    return {a: achados[a] for a in anos if a in achados}

In [5]:
def sha256(caminho: Path, bloco: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for b in iter(lambda: f.read(bloco), b""):
            h.update(b)
    return h.hexdigest()


def arquivo_valido(caminho: Path) -> bool:
    if not caminho.exists() or caminho.stat().st_size == 0:
        return False
    if caminho.suffix.lower() == ".parquet":
        try:
            pq.ParquetFile(caminho).metadata
            return True
        except Exception:  # noqa: BLE001
            return False
    with open(caminho, "rb") as f:
        return len(f.readline()) > 50  # cabeçalho plausível


def baixar(url: str, destino: Path, cfg: Config) -> dict:
    """Download com retomada por Range, backoff exponencial e verificação de tamanho."""
    parcial = destino.with_suffix(destino.suffix + ".part")
    total, last_mod = None, None
    for tentativa in range(1, cfg.tentativas_download + 1):
        ja = parcial.stat().st_size if parcial.exists() else 0
        hdr = dict(HDR, **({"Range": f"bytes={ja}-"} if ja else {}))
        try:
            with requests.get(url, headers=hdr, stream=True,
                              timeout=(cfg.timeout_conexao_s, cfg.timeout_leitura_s)) as r:
                if r.status_code == 416:            # já completo
                    break
                r.raise_for_status()
                if ja and r.status_code != 206:     # servidor ignorou Range: recomeça
                    ja = 0
                    parcial.unlink(missing_ok=True)
                tam = r.headers.get("Content-Length")
                total = (ja + int(tam)) if tam else None
                last_mod = r.headers.get("Last-Modified")
                with open(parcial, "ab" if ja else "wb") as f:
                    for b in r.iter_content(1 << 20):
                        f.write(b)
            if total is None or parcial.stat().st_size >= total:
                break
            log.warning("download incompleto (%d/%d bytes), retomando", parcial.stat().st_size, total)
        except (requests.RequestException, OSError) as e:
            espera = min(2 ** tentativa, 60)
            log.warning("tentativa %d/%d falhou (%s); nova tentativa em %ds",
                        tentativa, cfg.tentativas_download, type(e).__name__, espera)
            time.sleep(espera)
    else:
        raise RuntimeError(f"download não concluído após {cfg.tentativas_download} tentativas: {url}")
    parcial.replace(destino)
    if not arquivo_valido(destino):
        destino.unlink(missing_ok=True)
        raise RuntimeError(f"arquivo inválido após download: {destino.name}")
    return {"last_modified_servidor": last_mod, "bytes_esperados": total}


def obter_arquivos(cfg: Config) -> tuple[dict[int, Path], list[dict]]:
    manifesto_path = cfg.dir_raw / "manifesto_download.json"
    anterior = {m["ano"]: m for m in json.loads(manifesto_path.read_text(encoding="utf-8"))} if manifesto_path.exists() else {}
    caminhos: dict[int, Path] = {}
    manifesto: list[dict] = []
    urls: dict[str, dict[int, str]] = {}
    if not cfg.offline:
        pendentes = [a for a in cfg.anos if a not in cfg.arquivos_locais and a not in cfg.arquivos_manuais]
        for fmt in cfg.formatos:
            urls[fmt] = descobrir_urls(pendentes, fmt) if pendentes else {}

    for ano in cfg.anos:
        reg = {"ano": ano, "origem": None, "url": None, "arquivo": None}
        try:
            if ano in cfg.arquivos_locais:
                p = Path(cfg.arquivos_locais[ano])
                if not arquivo_valido(p):
                    raise FileNotFoundError(f"arquivo local inválido: {p}")
                reg.update(origem="local", arquivo=str(p))
            elif cfg.offline:
                cands = sorted((cfg.dir_raw / str(ano)).glob("INFLUD*"),
                               key=lambda q: data_publicacao(q.name) or pd.Timestamp(0))
                cands = [q for q in cands if q.suffix != ".part" and arquivo_valido(q)]
                if not cands:
                    raise FileNotFoundError(f"offline e sem arquivo em {cfg.dir_raw / str(ano)}")
                p = cands[-1]
                reg.update(origem="cache", arquivo=str(p))
            else:
                opcoes = ([("manual", cfg.arquivos_manuais[ano])] if ano in cfg.arquivos_manuais else []) + \
                         [("descoberto", urls[f][ano]) for f in cfg.formatos if ano in urls.get(f, {})] + \
                         ([("conhecido", URLS_CONHECIDAS[ano])] if ano in URLS_CONHECIDAS else [])
                erro = None
                for origem, url in dict.fromkeys(opcoes):
                    p = cfg.dir_raw / str(ano) / url.rsplit("/", 1)[-1]
                    p.parent.mkdir(parents=True, exist_ok=True)
                    try:
                        # nomes trazem a data de publicação ⇒ arquivo válido com mesmo nome é a mesma versão
                        sha_ant = anterior.get(ano, {}).get("sha256")
                        cache_ok = (not cfg.forcar_download and arquivo_valido(p)
                                    and (sha_ant is None or sha_ant == sha256(p)))
                        extra = {} if cache_ok else baixar(url, p, cfg)
                        reg.update(origem=origem if not cache_ok else f"{origem}/cache", url=url,
                                   arquivo=str(p), **extra)
                        break
                    except Exception as e:  # noqa: BLE001
                        erro = e
                        log.warning("[%d] %s falhou: %s", ano, origem, e)
                else:
                    raise RuntimeError(f"nenhuma fonte funcionou ({erro})")
            p = Path(reg["arquivo"])
            reg.update(bytes=p.stat().st_size, sha256=sha256(p),
                       data_publicacao=str(data_publicacao(p.name).date()) if data_publicacao(p.name) else None,
                       registrado_em_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"))
            caminhos[ano] = p
            log.info("[%d] ok: %s (%.0f MB, %s)", ano, p.name, reg["bytes"] / 1e6, reg["origem"])
        except Exception as e:  # noqa: BLE001
            reg["erro"] = str(e)
            log.error("[%d] INDISPONÍVEL: %s", ano, e)
        manifesto.append(reg)

    manifesto_path.write_text(json.dumps(manifesto, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    faltam = sorted(set(cfg.anos_obrigatorios) - set(caminhos))
    if faltam:
        raise RuntimeError(
            f"Anos obrigatórios ausentes: {faltam}. Baixe manualmente pela página do recurso e informe "
            f"em arquivos_manuais (url) ou arquivos_locais (caminho), ou remova de anos_obrigatorios "
            f"conscientemente (as semanas desses anos ficarão NaN, nunca zero).")
    return caminhos, manifesto

In [6]:
CAMINHOS, MANIFESTO = obter_arquivos(CFG)
pd.DataFrame(MANIFESTO)[["ano", "origem", "data_publicacao", "bytes", "arquivo"]]

2026-10-20 01:12:49,791 INFO [2019] ok: INFLUD19-23-03-2026.parquet (3 MB, cache)


2026-10-20 01:12:49,850 INFO [2020] ok: INFLUD20-23-03-2026.parquet (87 MB, cache)


2026-10-20 01:12:49,936 INFO [2021] ok: INFLUD21-23-03-2026.parquet (127 MB, cache)


2026-10-20 01:12:49,969 INFO [2022] ok: INFLUD22-23-03-2026.parquet (44 MB, cache)


2026-10-20 01:12:49,987 INFO [2023] ok: INFLUD23-23-03-2026.parquet (22 MB, cache)


2026-10-20 01:12:50,006 INFO [2024] ok: INFLUD24-23-03-2026.parquet (21 MB, cache)


2026-10-20 01:12:50,027 INFO [2025] ok: INFLUD25-19-10-2026.parquet (27 MB, cache)


2026-10-20 01:12:50,043 INFO [2026] ok: INFLUD26-19-10-2026.parquet (18 MB, cache)


,ano,origem,data_publicacao,bytes,arquivo
0,2019,cache,2026-03-23,3447406,data\raw\SRAG\2019\INFLUD19-23-03-2026.parquet
1,2020,cache,2026-03-23,87148508,data\raw\SRAG\2020\INFLUD20-23-03-2026.parquet
2,2021,cache,2026-03-23,127023754,data\raw\SRAG\2021\INFLUD21-23-03-2026.parquet
3,2022,cache,2026-03-23,43750567,data\raw\SRAG\2022\INFLUD22-23-03-2026.parquet
4,2023,cache,2026-03-23,22121151,data\raw\SRAG\2023\INFLUD23-23-03-2026.parquet
5,2024,cache,2026-03-23,21222050,data\raw\SRAG\2024\INFLUD24-23-03-2026.parquet
6,2025,cache,2026-10-19,26834893,data\raw\SRAG\2025\INFLUD25-19-10-2026.parquet
7,2026,cache,2026-10-19,18333232,data\raw\SRAG\2026\INFLUD26-19-10-2026.parquet


## 2. Leitura e harmonização

Lê só as colunas necessárias (mapeadas pelos candidatos de `COLUNAS`), mais as que casam com `regex_colunas_extras` (campos de imunização da ficha 2025/2026, se existirem). Tudo vira texto na leitura; tipagem acontece na limpeza. O filtro de idade é aplicado já na leitura, para caber em memória.

In [7]:
def _cod(s: pd.Series) -> pd.Series:
    """Código numérico inicial: '1', 1.0, '1-Sim' → 1; o resto → NaN (float)."""
    v = pd.to_numeric(s.astype("string").str.extract(r"^\s*(\d+)", expand=False), errors="coerce")
    return v.astype("float64")  # float puro: comparações devolvem bool numpy (sem NA de 3 valores)


def idade_meses(nu: pd.Series, tp: pd.Series) -> pd.Series:
    n, t = _cod(nu), _cod(tp)
    return pd.Series(np.select([t == 1, t == 2, t == 3], [n / 30.4375, n, n * 12], default=np.nan),
                     index=nu.index)


def colunas_arquivo(caminho: Path) -> tuple[list[str], dict]:
    if caminho.suffix.lower() == ".parquet":
        return list(pq.ParquetFile(caminho).schema_arrow.names), {}
    with open(caminho, encoding="latin-1", errors="replace") as f:
        cab = f.readline()
    kw = dict(sep=";" if cab.count(";") > cab.count(",") else ",", encoding="latin-1",
              encoding_errors="replace", dtype=str, low_memory=False)
    return list(pd.read_csv(caminho, nrows=0, **kw).columns), kw


def mapear_colunas(cols: list[str], cfg: Config) -> tuple[dict[str, str], list[str]]:
    norm = {c.upper().strip(): c for c in cols}
    mapa = {}
    for canon, cands in COLUNAS.items():
        for c in cands:
            if c in norm:
                mapa[canon] = norm[c]
                break
    extras = [norm[c] for c in norm if re.search(cfg.regex_colunas_extras, c)
              and norm[c] not in mapa.values()]
    return mapa, extras


def ler_ano(ano: int, caminho: Path, cfg: Config) -> tuple[pd.DataFrame, dict]:
    cols, kw = colunas_arquivo(caminho)
    mapa, extras = mapear_colunas(cols, cfg)
    falt = [c for c in OBRIGATORIAS if c not in mapa]
    if falt:
        raise KeyError(f"[{ano}] colunas obrigatórias ausentes: {falt}")
    if "PCR_VSR" not in mapa and "AN_VSR" not in mapa:
        raise KeyError(f"[{ano}] nenhuma coluna de VSR (PCR_VSR/AN_VSR); colunas com VSR: "
                       f"{[c for c in cols if 'VSR' in c.upper()]}")
    usar = list(dict.fromkeys(list(mapa.values()) + extras))
    inv = {v: k for k, v in mapa.items()}

    def _pos(df: pd.DataFrame) -> pd.DataFrame:
        df = df.rename(columns=inv).rename(columns={e: f"X_{e.upper()}" for e in extras})
        df = df.astype("string")
        meses = idade_meses(df["NU_IDADE_N"], df["TP_IDADE"])
        return df.loc[(meses >= 0) & (meses < cfg.idade_max_meses)]

    n_total = 0
    if caminho.suffix.lower() == ".parquet":
        pf = pq.ParquetFile(caminho)
        partes = []
        for i in range(pf.num_row_groups):
            t = pf.read_row_group(i, columns=usar).to_pandas()
            n_total += len(t)
            partes.append(_pos(t))
    else:
        partes = []
        for ch in pd.read_csv(caminho, usecols=usar, chunksize=250_000, **kw):
            n_total += len(ch)
            partes.append(_pos(ch))
    df = pd.concat(partes, ignore_index=True)
    for c in COLUNAS:
        if c not in df.columns:
            df[c] = pd.NA
    df["ARQUIVO_ANO"] = ano
    # data de publicação pelo nome; se o nome não a trouxer, máximo de DT_DIGITA do arquivo
    df["ARQUIVO_PUB"] = data_publicacao(caminho.name) or parse_data(df["DT_DIGITA"]).max()
    info = {"ano": ano, "linhas_arquivo": n_total, "linhas_menores2": len(df),
            "colunas_ausentes": sorted(set(COLUNAS) - set(mapa)),
            "colunas_extras": sorted(f"X_{e.upper()}" for e in extras),
            "mapeamento": {k: v for k, v in mapa.items() if k != v}}
    log.info("[%d] %d linhas, %d < %d meses; ausentes: %s", ano, n_total, len(df),
             cfg.idade_max_meses, info["colunas_ausentes"])
    return df, info

In [8]:
_lidos = [ler_ano(a, p, CFG) for a, p in sorted(CAMINHOS.items())]
BRUTO = pd.concat([d for d, _ in _lidos], ignore_index=True)
QC_COLUNAS = pd.DataFrame([i for _, i in _lidos]).set_index("ano")
del _lidos
QC_COLUNAS

2026-10-20 01:12:50,281 INFO [2019] 48941 linhas, 19298 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:12:54,852 INFO [2020] 1206920 linhas, 34498 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:01,016 INFO [2021] 1745672 linhas, 74049 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:03,064 INFO [2022] 560577 linhas, 84325 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:04,080 INFO [2023] 279453 linhas, 87778 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:05,041 INFO [2024] 267986 linhas, 89584 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:06,235 INFO [2025] 336439 linhas, 119660 < 24 meses; ausentes: ['CO_UNI_NOT']


2026-10-20 01:13:07,084 INFO [2026] 232824 linhas, 92290 < 24 meses; ausentes: ['CO_UNI_NOT']


,linhas_arquivo,linhas_menores2,colunas_ausentes,colunas_extras,mapeamento
ano,,,,,
2019,48941,19298,[CO_UNI_NOT],[X_CS_GESTANT],{}
2020,1206920,34498,[CO_UNI_NOT],[X_CS_GESTANT],{}
2021,1745672,74049,[CO_UNI_NOT],[X_CS_GESTANT],{}
2022,560577,84325,[CO_UNI_NOT],[X_CS_GESTANT],{}
2023,279453,87778,[CO_UNI_NOT],[X_CS_GESTANT],{}
2024,267986,89584,[CO_UNI_NOT],[X_CS_GESTANT],{}
2025,336439,119660,[CO_UNI_NOT],[X_CS_GESTANT],{}
2026,232824,92290,[CO_UNI_NOT],[X_CS_GESTANT],{}


## 3. Limpeza e definição de caso

Ordem fixa e auditada: datas → deduplicação entre arquivos (mantém a versão mais recente) → hospitalização → UF (residência, com recuperação pelo código IBGE do município) → marcadores laboratoriais → datas de disponibilidade. Cada etapa registra quantas linhas saíram em `qc_filtros.csv`.

In [9]:
def parse_data(s: pd.Series) -> pd.Series:
    """Aceita AAAA-MM-DD[ hh:mm:ss] e DD/MM/AAAA na mesma coluna."""
    s = s.astype("string").str.strip().str[:10]
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    br = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(br)


def inicio_semana(s: pd.Series) -> pd.Series:
    """Domingo da semana epidemiológica."""
    s = pd.to_datetime(s).dt.normalize()
    return s - pd.to_timedelta((s.dt.weekday + 1) % 7, unit="D")


def semana_epi(domingo: pd.Series) -> pd.DataFrame:
    """Ano e SE (padrão MMWR/MS): a semana pertence ao ano da sua quarta-feira."""
    qua = pd.to_datetime(domingo) + pd.Timedelta(days=3)
    return pd.DataFrame({"ano_epi": qua.dt.year.astype("Int64"),
                         "semana_epi": ((qua.dt.dayofyear - 1) // 7 + 1).astype("Int64")},
                        index=domingo.index)


def uf_valida(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip().str.upper()
    return s.where(s.isin(UFS))


def uf_por_municipio(cod: pd.Series) -> pd.Series:
    return _cod(cod).floordiv(10_000).map(COD_IBGE_UF).astype("string")

In [10]:
def limpar(bruto: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    d = bruto.copy()
    etapas = [("menores de 2 anos (leitura)", len(d))]
    corte_global = d["ARQUIVO_PUB"].max()

    for c in COLUNAS_DATA:
        d[c.lower()] = parse_data(d[c])
    d["idade_meses"] = idade_meses(d["NU_IDADE_N"], d["TP_IDADE"])

    # 1) data de sintomas válida e dentro da janela de estudo
    ini = pd.Timestamp(cfg.inicio_painel)
    ok = d["dt_sin_pri"].between(ini, corte_global)
    d = d.loc[ok]
    etapas.append(("data de sintomas válida", len(d)))

    # 2) deduplicação entre arquivos: mantém a publicação mais recente
    chave = [c for c in CHAVE_DEDUP if d[c].notna().any()]
    d = d.sort_values(["ARQUIVO_PUB", "ARQUIVO_ANO"])
    if "NU_NOTIFIC" in chave:
        sem_id = d["NU_NOTIFIC"].isna()
        d = pd.concat([d.loc[~sem_id].drop_duplicates(subset=chave, keep="last"), d.loc[sem_id]])
    else:
        log.warning("NU_NOTIFIC ausente: deduplicação por chave composta sem identificador")
        d = d.drop_duplicates(subset=chave + ["idade_meses", "DT_DIGITA"], keep="last")
    etapas.append(("após deduplicação " + "+".join(chave), len(d)))

    # 3) hospitalizados
    d["hospital"] = _cod(d["HOSPITAL"])
    d["hospital_ausente"] = d["hospital"].isna()
    if cfg.somente_hospitalizados:
        d = d.loc[(d["hospital"] == 1) | (cfg.incluir_hospital_ausente & d["hospital_ausente"])]
        etapas.append(("HOSPITAL = 1" + (" ou vazio" if cfg.incluir_hospital_ausente else ""), len(d)))

    # 4) UF: residência (com recuperação pelo município) e notificação
    d["uf_notificacao"] = uf_valida(d["SG_UF_NOT"]).fillna(uf_por_municipio(d["CO_MUN_NOT"]))
    uf_res = uf_valida(d["SG_UF"])
    d["uf_res_recuperada"] = uf_res.isna() & uf_por_municipio(d["CO_MUN_RES"]).notna()
    d["uf_residencia"] = uf_res.fillna(uf_por_municipio(d["CO_MUN_RES"]))
    d["uf_res_imputada_notif"] = d["uf_residencia"].isna() & d["uf_notificacao"].notna()
    d["uf_residencia"] = d["uf_residencia"].fillna(d["uf_notificacao"])
    # cada critério exige só a própria UF: sem UF de notificação, o caso sai apenas do painel por notificação
    d = d.loc[d["uf_residencia"].notna() | d["uf_notificacao"].notna()]
    etapas.append(("UF válida (residência ou notificação)", len(d)))

    # 5) marcadores laboratoriais
    pcr_vsr, an_vsr = _cod(d["PCR_VSR"]) == 1, _cod(d["AN_VSR"]) == 1
    d["vsr_pcr"], d["vsr_an"] = pcr_vsr, an_vsr
    d["vsr"] = pcr_vsr | an_vsr
    # VSR positivo conta mesmo com PCR_RESUL/RES_AN fora de {1,2} (~10 casos/ano, codificação inconsistente)
    d["lab_resultado"] = d["vsr"] | _cod(d["PCR_RESUL"]).isin([1, 2]) | _cod(d["RES_AN"]).isin([1, 2])
    d["testado_painel"] = d["vsr"] | _cod(d["POS_PCROUT"]).isin([1, 2]) | _cod(d["POS_AN_OUT"]).isin([1, 2])
    d["sars2"] = _cod(d["PCR_SARS2"]) == 1
    d["classi_fin"] = _cod(d["CLASSI_FIN"]).astype("Int64")
    d["uti"] = (_cod(d["UTI"]) == 1)
    d["obito"] = (_cod(d["EVOLUCAO"]) == 2)
    for c in ("vsr", "vsr_pcr", "vsr_an", "lab_resultado", "testado_painel", "sars2", "uti", "obito"):
        d[c] = d[c].astype(bool)

    # 6) datas de disponibilidade (anos com DT_DIGITA ausente → data de notificação, marcado)
    pub = d["ARQUIVO_PUB"]
    dig = d["dt_digita"].where(d["dt_digita"].between(d["dt_sin_pri"] - pd.Timedelta(days=7), pub))
    d["disp_reg_imputada"] = dig.isna()
    dig = dig.fillna(d["dt_notific"].where(d["dt_notific"].between(d["dt_sin_pri"], pub)))
    d["dt_disp_registro"] = dig
    res = pd.concat([d["dt_pcr"].where(pcr_vsr), d["dt_res_an"].where(an_vsr)], axis=1).min(axis=1)
    res = res.where(res.between(d["dt_sin_pri"], pub))
    d["dt_resultado_vsr"] = res
    d["disp_vsr_sem_data_res"] = d["vsr"] & res.isna()
    d["dt_disp_vsr"] = pd.concat([dig, res], axis=1).max(axis=1).where(d["vsr"])

    # 7) calendário
    d["semana"] = inicio_semana(d["dt_sin_pri"])
    d = d.join(semana_epi(d["semana"]))
    d["semana_disp_registro"] = inicio_semana(d["dt_disp_registro"])
    d["semana_disp_vsr"] = inicio_semana(d["dt_disp_vsr"])
    lim = cfg.faixas_meses
    d["faixa_etaria"] = pd.cut(d["idade_meses"], bins=list(lim), right=False,
                               labels=[f"{a:02d}-{b - 1:02d}m" for a, b in zip(lim[:-1], lim[1:])]).astype("string")

    manter = ["ARQUIVO_ANO", "ARQUIVO_PUB", "NU_NOTIFIC", "dt_sin_pri", "dt_notific", "dt_digita", "dt_pcr",
              "dt_res_an", "dt_encerra", "dt_disp_registro", "dt_resultado_vsr", "dt_disp_vsr",
              "semana", "ano_epi", "semana_epi", "semana_disp_registro", "semana_disp_vsr",
              "idade_meses", "faixa_etaria", "uf_residencia", "uf_notificacao", "uf_res_recuperada",
              "uf_res_imputada_notif", "CO_MUN_RES", "vsr", "vsr_pcr", "vsr_an", "lab_resultado",
              "testado_painel", "sars2", "classi_fin", "uti", "obito", "hospital_ausente", "disp_reg_imputada",
              "disp_vsr_sem_data_res"] + [c for c in d.columns if c.startswith("X_")]
    d = d[manter].rename(columns={"NU_NOTIFIC": "nu_notific", "CO_MUN_RES": "co_mun_res",
                                  "ARQUIVO_ANO": "arquivo_ano", "ARQUIVO_PUB": "arquivo_pub"})
    d = d.sort_values(["semana", "uf_residencia"]).reset_index(drop=True)
    qc = pd.DataFrame(etapas, columns=["etapa", "linhas"])
    qc["removidas"] = -qc["linhas"].diff().fillna(0).astype(int)
    return d, qc

In [11]:
CASOS, QC_FILTROS = limpar(BRUTO, CFG)
del BRUTO
# Data de corte = última digitação válida; o painel termina na última semana completa (sábado ≤ corte)
DATA_CORTE = CASOS["dt_disp_registro"].max()
SEMANA_FINAL = inicio_semana(pd.Series([DATA_CORTE + pd.Timedelta(days=1)])).iloc[0] - pd.Timedelta(weeks=1)
CASOS = CASOS.loc[CASOS["semana"] <= SEMANA_FINAL].reset_index(drop=True)
QC_FILTROS.loc[len(QC_FILTROS)] = ["semana completa (sábado ≤ data de corte)", len(CASOS),
                                   QC_FILTROS["linhas"].iloc[-1] - len(CASOS)]
log.info("data de corte %s; última semana completa %s", DATA_CORTE.date(), SEMANA_FINAL.date())
display(QC_FILTROS)

QC_ANO = (CASOS.groupby("ano_epi")
          .agg(casos=("vsr", "size"), vsr=("vsr", "sum"), lab=("lab_resultado", "mean"),
               painel=("testado_painel", "mean"), sem_data_res_vsr=("disp_vsr_sem_data_res", "sum"),
               disp_reg_imputada=("disp_reg_imputada", "mean"), uf_res_imput=("uf_res_imputada_notif", "mean"),
               hospital_ausente=("hospital_ausente", "sum"),
               sem_uf_notificacao=("uf_notificacao", lambda s: int(s.isna().sum())))
          .assign(positividade_painel=lambda x: x["vsr"] / (x["painel"] * x["casos"]),
                  positividade_lab=lambda x: x["vsr"] / (x["lab"] * x["casos"])))
display(QC_ANO.round(3))
# Arquivo de origem × ano de sintomas: mostra se os arquivos são por ano de sintomas ou de notificação
display(pd.crosstab(CASOS["arquivo_ano"], CASOS["ano_epi"]))

2026-10-20 01:13:14,093 INFO data de corte 2026-10-18; última semana completa 2026-10-11


,etapa,linhas,removidas
0,menores de 2 anos (leitura),601482,0
1,data de sintomas válida,601482,0
2,após deduplicação NU_NOTIFIC+DT_NOTIFIC+SG_UF_...,601482,0
3,HOSPITAL = 1,585233,16249
4,UF válida (residência ou notificação),585233,0
5,semana completa (sábado ≤ data de corte),585232,1


,casos,vsr,lab,painel,sem_data_res_vsr,disp_reg_imputada,uf_res_imput,hospital_ausente,sem_uf_notificacao,positividade_painel,positividade_lab
ano_epi,,,,,,,,,,,
2019,19122,5029,0.851,0.388,1,0.401,0.000,0,9,0.678,0.309
2020,33260,1120,0.858,0.175,0,0.000,0.000,0,21,0.193,0.039
2021,71859,11126,0.882,0.265,0,0.001,0.000,0,18,0.583,0.176
2022,82388,14977,0.911,0.373,1,0.001,0.000,0,24,0.487,0.200
2023,85037,23419,0.926,0.463,1,0.000,0.000,0,5,0.595,0.297
2024,87134,27162,0.932,0.556,5,0.000,0.000,0,0,0.561,0.335
2025,116698,43237,0.932,0.637,10,0.000,0.000,0,64,0.581,0.398
2026,89734,32340,0.910,0.631,14,0.000,0.001,0,50,0.571,0.396


ano_epi,2019,2020,2021,2022,2023,2024,2025,2026
arquivo_ano,,,,,,,,
2019,19122,0,0,0,0,0,0,0
2020,0,33260,0,0,0,0,0,0
2021,0,0,71859,0,0,0,0,0
2022,0,0,0,82388,0,0,0,0
2023,0,0,0,0,85037,0,0,0
2024,0,0,0,0,0,87134,0,0
2025,0,0,0,0,0,0,116698,0
2026,0,0,0,0,0,0,0,89734


## 4. Cobertura temporal e períodos

Semana sem arquivo-fonte vira `NaN`, nunca zero. A cobertura da semana $k$ do ano $Y$ é $\sum_\delta P(\text{arquivo } Y{+}\delta \mid k)\,\mathbb 1[\text{arquivo } Y{+}\delta \text{ presente}]$, com $P$ estimada nos anos cujos arquivos vizinhos estão todos presentes (captura casos de dezembro publicados no arquivo do ano seguinte, e vice-versa). Semanas com cobertura < 0,99 ficam `coberta = False`.

In [12]:
def padrao_arquivos(casos: pd.DataFrame, cfg: Config, presentes: set[int]) -> pd.DataFrame:
    """P(caso da SE k do ano Y está no arquivo Y + δ), estimado nos anos com vizinhos presentes."""
    x = casos.assign(delta=casos["arquivo_ano"] - casos["ano_epi"].astype(int))
    completos = [a for a in presentes if {a - 1, a, a + 1} & set(cfg.anos) <= presentes]
    x = x.loc[x["ano_epi"].isin(completos)]
    if x.empty:
        return pd.DataFrame({"semana_epi": range(1, 54), "delta": 0, "p": 1.0})
    return (pd.crosstab(x["semana_epi"], x["delta"], normalize="index")
            .reindex(range(1, 54)).ffill().bfill()
            .stack().rename("p").reset_index())


def calendario(cfg: Config, casos: pd.DataFrame, caminhos: dict[int, Path]) -> pd.DataFrame:
    fim = casos["semana"].max()
    sem = pd.DataFrame({"semana": pd.date_range(cfg.inicio_painel, fim, freq="7D")})
    sem = sem.join(semana_epi(sem["semana"]))
    presentes = set(caminhos)
    pad = padrao_arquivos(casos, cfg, presentes)
    m = sem[["semana", "ano_epi", "semana_epi"]].merge(pad, on="semana_epi", how="left")
    # arquivo fora do intervalo publicado (ex.: 2018, 2027) não conta como ausente
    arq = m["ano_epi"].astype(int) + m["delta"].astype(int)
    m["presente"] = arq.isin(presentes) | ~arq.isin(cfg.anos)
    cob = (m["p"] * m["presente"]).groupby(m["semana"]).sum()
    sem["cobertura_arquivos"] = sem["semana"].map(cob).fillna(0.0).clip(0, 1)
    sem["coberta"] = sem["cobertura_arquivos"] >= 0.99
    q_ini, q_fim = pd.Timestamp(cfg.quebra_ini), pd.Timestamp(cfg.quebra_fim)
    imu = pd.Timestamp(cfg.imunizacao_ini)
    sem["periodo"] = np.select(
        [sem["semana"] < q_ini, sem["semana"] <= q_fim, sem["semana"] < imu],
        ["pre_pandemia", "quebra_npi", "pos_pandemia"], default="imunizacao")
    sem["quebra_npi"] = sem["periodo"].eq("quebra_npi")
    sem["imunizacao"] = sem["periodo"].eq("imunizacao")
    sem["vacina_materna"] = (sem["semana"] >= pd.Timestamp(cfg.vacina_materna_ini)
                             if cfg.vacina_materna_ini else False)
    return sem


CAL = calendario(CFG, CASOS, CAMINHOS)
display(CAL.groupby("ano_epi").agg(semanas=("semana", "size"), coberta=("coberta", "mean"),
                                   periodo=("periodo", lambda s: ",".join(sorted(set(s))))))

,semanas,coberta,periodo
ano_epi,,,
2019,52,1.0,pre_pandemia
2020,53,1.0,"pre_pandemia,quebra_npi"
2021,52,1.0,quebra_npi
2022,52,1.0,pos_pandemia
2023,52,1.0,pos_pandemia
2024,52,1.0,pos_pandemia
2025,53,1.0,pos_pandemia
2026,41,1.0,"imunizacao,pos_pandemia"


## 5. Painel semanal hierárquico

Grão: `criterio_uf` × `nivel` × `unidade` × `faixa_etaria` × `semana`. As contagens são feitas por UF e faixa; região, Brasil e a faixa `00-23m` são somas, logo a hierarquia é coerente por construção ($\mathbf y = S\,\mathbf y_{\text{UF}}$).

In [13]:
FAIXA_TOTAL = f"00-{CFG.idade_max_meses - 1:02d}m"
SERIES_CONTAGEM = {  # nome: coluna booleana em CASOS (None = todos)
    "srag": None, "vsr": "vsr", "vsr_pcr": "vsr_pcr", "vsr_an": "vsr_an",
    "lab_resultado": "lab_resultado", "testado_painel": "testado_painel", "sars2": "sars2",
    "vsr_uti": ("vsr", "uti"), "vsr_obito": ("vsr", "obito"),
}


def contar_uf(casos: pd.DataFrame, col_uf: str, cal: pd.DataFrame) -> pd.DataFrame:
    """Contagens completas (todas as semanas × UFs × faixas, zeros explícitos)."""
    x = casos.assign(uf=casos[col_uf])
    faixas = sorted(x["faixa_etaria"].dropna().unique())
    idx = pd.MultiIndex.from_product([cal["semana"], UFS, faixas], names=["semana", "uf", "faixa_etaria"])
    out = pd.DataFrame(index=idx)
    for nome, col in SERIES_CONTAGEM.items():
        if col is None:
            m = pd.Series(True, index=x.index)
        elif isinstance(col, tuple):
            m = x[list(col)].all(axis=1)
        else:
            m = x[col]
        out[nome] = x.loc[m].groupby(["semana", "uf", "faixa_etaria"]).size()
    return out.fillna(0).astype("int64").reset_index()


LIMITES_VSR_EST = {"vsr_est_inf": "lab_resultado", "vsr_est_sup": "testado_painel"}  # série: denominador


def positividade_corrigida(painel_uf: pd.DataFrame, cfg: Config, saida: str, denominador: str) -> pd.DataFrame:
    """saida = srag × p̃ (trailing, shrinkage para a região), por UF × faixa. Só usa passado."""
    w, k = cfg.janela_positividade, cfg.kappa_shrinkage
    p = painel_uf.sort_values(["uf", "faixa_etaria", "semana"]).copy()
    p["regiao"] = p["uf"].map(REGIAO_UF)
    roll = lambda g: g.rolling(w, min_periods=1).sum()  # noqa: E731
    g = p.groupby(["uf", "faixa_etaria"], sort=False)
    p["y_w"] = g["vsr"].transform(roll)
    p["n_w"] = g[denominador].transform(roll)
    reg = (p.groupby(["regiao", "faixa_etaria", "semana"])[["y_w", "n_w"]].sum()
           .rename(columns={"y_w": "yr", "n_w": "nr"}))
    p = p.join(reg, on=["regiao", "faixa_etaria", "semana"])
    p_reg = (p["yr"] / p["nr"]).where(p["nr"] > 0)
    # Região sem testagem na janela: positividade nacional da faixa
    br = p.groupby(["faixa_etaria", "semana"])[["y_w", "n_w"]].transform("sum")
    p_reg = p_reg.fillna((br["y_w"] / br["n_w"]).where(br["n_w"] > 0))
    num = p["y_w"] + k * p_reg.fillna(0)
    den = p["n_w"] + k * p_reg.notna()
    p["positividade_corrigida"] = (num / den).where(den > 0)
    # Nunca abaixo do confirmado
    p[saida] = np.maximum((p["srag"] * p["positividade_corrigida"]).fillna(0.0), p["vsr"])
    return p.drop(columns=["y_w", "n_w", "yr", "nr", "regiao", "positividade_corrigida"])


def agregar_hierarquia(p_uf: pd.DataFrame, valores: list[str]) -> pd.DataFrame:
    p = p_uf.copy()
    tot = p.groupby(["semana", "uf"], as_index=False)[valores].sum().assign(faixa_etaria=FAIXA_TOTAL)
    p = pd.concat([p, tot], ignore_index=True)
    p["regiao"] = p["uf"].map(REGIAO_UF)
    uf = p.rename(columns={"uf": "unidade"}).assign(nivel="UF")
    reg = (p.groupby(["semana", "regiao", "faixa_etaria"], as_index=False)[valores].sum()
           .rename(columns={"regiao": "unidade"}).assign(nivel="REGIAO"))
    br = (p.groupby(["semana", "faixa_etaria"], as_index=False)[valores].sum()
          .assign(unidade="BR", nivel="BRASIL"))
    return pd.concat([br, reg, uf.drop(columns="regiao")], ignore_index=True)


def montar_painel(casos: pd.DataFrame, cal: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    partes, ufs_base = [], []
    for crit in cfg.criterios_uf:
        col = {"residencia": "uf_residencia", "notificacao": "uf_notificacao"}[crit]
        p_uf = contar_uf(casos, col, cal)
        for saida, den in LIMITES_VSR_EST.items():
            p_uf = positividade_corrigida(p_uf, cfg, saida, den)
        ufs_base.append(p_uf.assign(criterio_uf=crit))
        valores = list(SERIES_CONTAGEM) + list(LIMITES_VSR_EST)
        partes.append(agregar_hierarquia(p_uf, valores).assign(criterio_uf=crit))
    painel = pd.concat(partes, ignore_index=True)
    painel = painel.merge(cal, on="semana", how="left")
    # Semanas sem arquivo-fonte: NaN (nunca zero)
    valores = list(SERIES_CONTAGEM) + list(LIMITES_VSR_EST)
    painel[valores] = painel[valores].astype("float64")
    painel.loc[~painel["coberta"], valores] = np.nan
    painel["positividade_painel"] = (painel["vsr"] / painel["testado_painel"]).where(painel["testado_painel"] > 0)
    painel["positividade_lab"] = (painel["vsr"] / painel["lab_resultado"]).where(painel["lab_resultado"] > 0)
    painel["frac_testado_painel"] = (painel["testado_painel"] / painel["srag"]).where(painel["srag"] > 0)
    return painel, pd.concat(ufs_base, ignore_index=True)


PAINEL, PAINEL_UF_BASE = montar_painel(CASOS, CAL, CFG)
PAINEL.shape

(107448, 27)

## 6. Atraso de disponibilidade e completude

Triângulo $z_{u,f,t,d}$ = casos com sintomas na semana $t$ disponíveis $d$ semanas depois (VSR pela data de resultado; SRAG pela digitação). Atrasos negativos viram 0 (contados em QC); $d \ge D$ vai para o último *bucket*. A versão conhecida na origem $o$ é
$$y^{(o)}_{u,t} = \sum_{d=0}^{\lfloor (o-t)/7\rfloor} z_{u,t,d}.$$
$F_r(d)$ é estimada por região com as últimas temporadas já maturadas e define `completude_esperada` e `semana_incompleta` no painel (relativas à data de corte).

In [14]:
def triangulo(casos: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, dict]:
    D = cfg.atraso_max_semanas
    partes, qc = [], {}
    for serie, sem_disp, filtro in (("srag", "semana_disp_registro", None), ("vsr", "semana_disp_vsr", "vsr")):
        x = casos if filtro is None else casos.loc[casos[filtro]]
        atraso = ((x[sem_disp] - x["semana"]).dt.days // 7)
        qc[f"{serie}_atraso_negativo"] = int((atraso < 0).sum())
        qc[f"{serie}_sem_data_disp"] = int(atraso.isna().sum())
        atraso = atraso.clip(lower=0).fillna(D).clip(upper=D).astype(int)   # sem data → bucket final
        for crit in cfg.criterios_uf:
            col = {"residencia": "uf_residencia", "notificacao": "uf_notificacao"}[crit]
            t = (x.assign(atraso=atraso, uf=x[col])
                 .groupby(["uf", "faixa_etaria", "semana", "atraso"]).size().rename("casos").reset_index())
            partes.append(t.assign(serie=serie, criterio_uf=crit))
    tri = pd.concat(partes, ignore_index=True)
    tri["atraso_e_bucket_final"] = tri["atraso"] == D
    return tri, qc


def completude(tri: pd.DataFrame, cfg: Config, data_corte: pd.Timestamp) -> pd.DataFrame:
    """F_r(d): fração acumulada por região e série, só semanas totalmente maturadas (t + D ≤ corte)."""
    D = cfg.atraso_max_semanas
    crit = cfg.criterios_uf[0]
    x = tri.loc[(tri["criterio_uf"] == crit) & (tri["semana"] <= data_corte - pd.Timedelta(weeks=D))].copy()
    anos = sorted(x["semana"].dt.year.unique())[-cfg.temporadas_completude:]
    x = x.loc[x["semana"].dt.year.isin(anos)]
    x["regiao"] = x["uf"].map(REGIAO_UF)
    frames = []
    for (serie, reg), g in pd.concat([x, x.assign(regiao="BR")]).groupby(["serie", "regiao"]):
        h = g.groupby("atraso")["casos"].sum().reindex(range(D + 1), fill_value=0)
        frames.append(pd.DataFrame({"serie": serie, "regiao": reg, "atraso": range(D + 1),
                                    "F": (h.cumsum() / h.sum()).to_numpy(), "n": int(h.sum()),
                                    "anos_base": ",".join(map(str, anos))}))
    return pd.concat(frames, ignore_index=True)


def anexar_completude(painel: pd.DataFrame, comp: pd.DataFrame, data_corte: pd.Timestamp,
                      cfg: Config) -> pd.DataFrame:
    D = cfg.atraso_max_semanas
    p = painel.copy()
    p["atraso_ate_corte"] = ((inicio_semana(pd.Series([data_corte])).iloc[0] - p["semana"]).dt.days // 7).clip(0, D)
    p["regiao_ref"] = np.where(p["nivel"] == "UF", p["unidade"].map(REGIAO_UF),
                               np.where(p["nivel"] == "REGIAO", p["unidade"], "BR"))
    for serie in ("vsr", "srag"):
        f = comp.loc[comp["serie"] == serie].set_index(["regiao", "atraso"])["F"]
        p[f"completude_esperada_{serie}"] = pd.MultiIndex.from_arrays(
            [p["regiao_ref"], p["atraso_ate_corte"]]).map(f).astype("float64")
    p["semana_incompleta"] = p["completude_esperada_vsr"] < cfg.completude_flag
    return p.drop(columns=["regiao_ref"])


TRI, QC_ATRASO = triangulo(CASOS, CFG)
COMP = completude(TRI, CFG, DATA_CORTE)
PAINEL = anexar_completude(PAINEL, COMP, DATA_CORTE, CFG)
log.info("data de corte (máx. DT_DIGITA válida): %s | QC atraso: %s", DATA_CORTE.date(), QC_ATRASO)
display(COMP.pivot_table(index=["serie", "regiao"], columns="atraso", values="F").iloc[:, :9].round(2))

2026-10-20 01:13:15,765 INFO data de corte (máx. DT_DIGITA válida): 2026-10-18 | QC atraso: {'srag_atraso_negativo': 30, 'srag_sem_data_disp': 0, 'vsr_atraso_negativo': 0, 'vsr_sem_data_disp': 0}


atraso                 0     1     2     3     4     5     6     7     8
serie regiao                                                            
srag  BR            0.18  0.65  0.80  0.87  0.91  0.93  0.95  0.96  0.96
      CENTRO_OESTE  0.17  0.67  0.85  0.91  0.94  0.95  0.97  0.97  0.98
      NORDESTE      0.22  0.69  0.83  0.89  0.92  0.94  0.96  0.96  0.97
      NORTE         0.20  0.67  0.82  0.88  0.92  0.94  0.95  0.96  0.97
      SUDESTE       0.15  0.59  0.76  0.84  0.89  0.92  0.94  0.95  0.96
      SUL           0.21  0.71  0.84  0.89  0.92  0.93  0.95  0.95  0.96
vsr   BR            0.09  0.57  0.79  0.87  0.91  0.94  0.95  0.96  0.97
      CENTRO_OESTE  0.10  0.63  0.84  0.90  0.92  0.94  0.96  0.97  0.98
      NORDESTE      0.11  0.58  0.79  0.88  0.92  0.94  0.96  0.97  0.97
      NORTE         0.07  0.59  0.82  0.90  0.94  0.95  0.97  0.98  0.98
      SUDESTE       0.09  0.52  0.75  0.84  0.89  0.92  0.94  0.95  0.96
      SUL           0.10  0.62  0.84  0.91  0.93  0.95  0.96  0.96  0.97

## 7. Temporadas, climatologia e aproximações para 2020–2021

- `temporada`: ano epidemiológico, ou deslocado se `semana_inicio_temporada[regiao]` for definido (para temporadas que cruzam a virada do ano). `semana_temporada` vai de 1 a 53, sem achatar a SE 53; na climatologia, a semana 53 usa só as temporadas que a têm e, se nenhuma tiver, a média das semanas 52 e 1 (`n_temporadas = 0`).
- Climatologia por unidade × faixa × série × `semana_temporada` sobre $\log(1+y)$ e sobre a fração da temporada $y_t/\sum_t y_t$:
  - `tipo = "expanding"`: para cada `temporada_alvo` $S$, usa só temporadas de referência $< S$ (sem vazamento, para modelos);
  - `tipo = "referencia"`: todas as temporadas de referência (fora 2020–21 e da era de imunização) — EDA e sensibilidade.
- Imputação do período `quebra_npi` (sensibilidade; os valores observados nunca são sobrescritos):
  $\;y^{\text{imp}}_t = \exp\big(\overline{\log(1+y)}_{k(t)}\big) - 1$ em `*_imput_ref`, e o mesmo com a climatologia *expanding* da temporada em `*_imput_passado` (para 2020–21 isso significa só 2019).

In [15]:
SERIES_CLIM = ["vsr", "srag"] + list(LIMITES_VSR_EST)


def anexar_temporada(painel: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    p = painel.copy()
    reg = np.where(p["nivel"] == "UF", p["unidade"].map(REGIAO_UF),
                   np.where(p["nivel"] == "REGIAO", p["unidade"], "BR"))
    inicio = pd.Series(reg, index=p.index).map(cfg.semana_inicio_temporada).fillna(1).astype(int)
    ano, se = p["ano_epi"].astype(int), p["semana_epi"].astype(int)
    p["temporada"] = np.where(se >= inicio, ano, ano - 1)
    # semana dentro da temporada (1..53), contada a partir do domingo da SE `inicio` do ano da temporada
    jan1 = pd.to_datetime(pd.Series(p["temporada"], index=p.index).astype(str) + "-01-01")
    se1 = jan1 + pd.to_timedelta((2 - jan1.dt.weekday) % 7 - 3, unit="D")   # 1ª quarta-feira − 3 dias
    p["semana_temporada"] = ((p["semana"] - se1).dt.days // 7 - (inicio - 1) + 1).astype(int)
    return p


def temporadas_referencia(painel: pd.DataFrame, cfg: Config) -> list[int]:
    ok = (painel.loc[painel["nivel"] == "BRASIL"].groupby("temporada")
          .agg(cob=("coberta", "mean"), n=("semana", "nunique"), imu=("imunizacao", "any")))
    ok = ok.loc[(ok["cob"] >= cfg.cobertura_min_temporada) & (ok["n"] >= 52) & ~ok["imu"]]
    return sorted(set(ok.index) - set(cfg.temporadas_excluidas_climatologia))


def climatologia(painel: pd.DataFrame, refs: list[int], cfg: Config) -> pd.DataFrame:
    chaves = ["criterio_uf", "nivel", "unidade", "faixa_etaria"]
    base = painel.loc[painel["temporada"].isin(refs), chaves + ["temporada", "semana_temporada"] + SERIES_CLIM]
    long = base.melt(id_vars=chaves + ["temporada", "semana_temporada"], value_vars=SERIES_CLIM,
                     var_name="serie", value_name="y").dropna(subset=["y"])
    long = long.groupby(chaves + ["serie", "temporada", "semana_temporada"], as_index=False)["y"].mean()
    long["log1p"] = np.log1p(long["y"])
    tot = long.groupby(chaves + ["serie", "temporada"])["y"].transform("sum")
    long["fracao"] = (long["y"] / tot).where(tot > 0)

    def resumir(df: pd.DataFrame) -> pd.DataFrame:
        r = (df.groupby(chaves + ["serie", "semana_temporada"])
             .agg(media_log1p=("log1p", "mean"), dp_log1p=("log1p", "std"),
                  media_fracao=("fracao", "mean"), n_temporadas=("temporada", "nunique"))
             .reset_index())
        # SE 53 sem temporada que a tenha: média das semanas 52 e 1 (n_temporadas = 0)
        k = chaves + ["serie"]
        viz = r.loc[r["semana_temporada"].isin([52, 1])].groupby(k, as_index=False)[
            ["media_log1p", "dp_log1p", "media_fracao"]].mean()
        com53 = r.loc[r["semana_temporada"] == 53, k].assign(_tem=True)
        viz = viz.merge(com53, on=k, how="left")
        viz = viz.loc[viz["_tem"].isna()].drop(columns="_tem").assign(semana_temporada=53, n_temporadas=0)
        return pd.concat([r, viz], ignore_index=True)

    alvos = sorted(set(painel["temporada"]))
    frames = [resumir(long).assign(tipo="referencia", temporada_alvo=-1,
                                   temporadas_usadas=",".join(map(str, refs)))]
    for s in alvos:
        usadas = [r for r in refs if r < s]
        if usadas:
            frames.append(resumir(long.loc[long["temporada"].isin(usadas)])
                          .assign(tipo="expanding", temporada_alvo=s, temporadas_usadas=",".join(map(str, usadas))))
    return pd.concat(frames, ignore_index=True)


def imputar_quebra(painel: pd.DataFrame, clim: pd.DataFrame) -> pd.DataFrame:
    chaves = ["criterio_uf", "nivel", "unidade", "faixa_etaria", "semana_temporada"]
    p = painel.copy()
    for s in SERIES_CLIM:
        ref = clim.loc[(clim["tipo"] == "referencia") & (clim["serie"] == s)].set_index(chaves)["media_log1p"]
        exp_ = clim.loc[(clim["tipo"] == "expanding") & (clim["serie"] == s)].set_index(
            chaves + ["temporada_alvo"])["media_log1p"]
        k_ref = pd.MultiIndex.from_frame(p[chaves])
        k_exp = pd.MultiIndex.from_frame(p[chaves].assign(temporada_alvo=p["temporada"]))
        v_ref = np.expm1(k_ref.map(ref).astype("float64"))
        v_exp = np.expm1(k_exp.map(exp_).astype("float64"))
        q = p["quebra_npi"].to_numpy()
        p[f"{s}_imput_ref"] = np.where(q, v_ref, p[s])
        p[f"{s}_imput_passado"] = np.where(q, v_exp, p[s])
    return p


PAINEL = anexar_temporada(PAINEL, CFG)
REFS = temporadas_referencia(PAINEL, CFG)
log.info("temporadas de referência (climatologia): %s", REFS)
CLIM = climatologia(PAINEL, REFS, CFG)
PAINEL = imputar_quebra(PAINEL, CLIM)

2026-10-20 01:13:15,871 INFO temporadas de referência (climatologia): [2019, 2022, 2023, 2024, 2025]


## 8. Hierarquia, tiers de UF e população (opcional)

In [16]:
def tabela_hierarquia() -> tuple[pd.DataFrame, pd.DataFrame]:
    unidades = pd.DataFrame(
        [("BR", "BRASIL", None)] + [(r, "REGIAO", "BR") for r in REGIOES] +
        [(u, "UF", REGIAO_UF[u]) for u in sorted(UFS, key=lambda u: (REGIOES.index(REGIAO_UF[u]), u))],
        columns=["unidade", "nivel", "pai"])
    ufs = unidades.loc[unidades["nivel"] == "UF", "unidade"].tolist()
    S = pd.DataFrame(0, index=unidades["unidade"], columns=ufs, dtype="int8")
    S.loc["BR", :] = 1
    for u in ufs:
        S.loc[REGIAO_UF[u], u] = 1
        S.loc[u, u] = 1
    return unidades, S


def classificar_ufs(painel: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    x = painel.loc[(painel["criterio_uf"] == cfg.criterios_uf[0]) & (painel["nivel"] == "UF")
                   & (painel["faixa_etaria"] == FAIXA_TOTAL)]
    # soma sobre semanas cobertas (temporadas parcialmente cobertas ficam levemente subestimadas)
    tab = x.pivot_table(index="unidade", columns="temporada", values="vsr", aggfunc="sum")
    cob = x.groupby("temporada")["coberta"].mean()
    disp = [t for t in cfg.tier_temporadas if t in tab.columns and cob.get(t, 0) >= cfg.cobertura_min_temporada]
    if len(disp) < len(cfg.tier_temporadas):
        log.warning("tiers calculados sem temporadas %s (ausentes ou incompletas)",
                    sorted(set(cfg.tier_temporadas) - set(disp)))
    m = tab[disp].min(axis=1)
    out = tab.copy()
    out.columns = [f"vsr_{c}" for c in out.columns]
    out["min_temporadas_tier"] = m
    out["temporadas_tier"] = ",".join(map(str, disp))
    out["tier"] = np.select([m >= cfg.tier_a_min, m >= cfg.tier_b_min], ["A", "B"], default="C")
    out["regiao"] = out.index.map(REGIAO_UF)
    return out.reset_index().sort_values(["tier", "min_temporadas_tier"], ascending=[True, False])


def anexar_populacao(painel: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Se cfg.arquivo_populacao existir (colunas: uf, ano, pop_menores2), adiciona pop e taxas por 100 mil."""
    if not cfg.arquivo_populacao or not Path(cfg.arquivo_populacao).exists():
        log.info("população não informada: taxas não calculadas (preencha arquivo_populacao)")
        return painel
    pop = pd.read_csv(cfg.arquivo_populacao)
    pop["regiao"] = pop["uf"].map(REGIAO_UF)
    niveis = pd.concat([
        pop.rename(columns={"uf": "unidade"})[["unidade", "ano", "pop_menores2"]],
        pop.groupby(["regiao", "ano"], as_index=False)["pop_menores2"].sum().rename(columns={"regiao": "unidade"}),
        pop.groupby("ano", as_index=False)["pop_menores2"].sum().assign(unidade="BR")])
    p = painel.merge(niveis.rename(columns={"ano": "ano_epi"}), on=["unidade", "ano_epi"], how="left")
    p.loc[p["faixa_etaria"] != FAIXA_TOTAL, "pop_menores2"] = np.nan   # pop só para o total etário
    for s in ["vsr", "srag"] + list(LIMITES_VSR_EST):
        p[f"{s}_taxa100k"] = 1e5 * p[s] / p["pop_menores2"]
    return p


UNIDADES, MATRIZ_S = tabela_hierarquia()
TIERS = classificar_ufs(PAINEL, CFG)
PAINEL = anexar_populacao(PAINEL, CFG)
display(TIERS)

,unidade,vsr_2019,vsr_2020,vsr_2021,vsr_2022,vsr_2023,vsr_2024,vsr_2025,vsr_2026,min_temporadas_tier,temporadas_tier,tier,regiao
25,SP,786.0,407.0,4515.0,5214.0,6041.0,6843.0,10205.0,8021.0,6041.0,"2023,2024,2025",A,SUDESTE
17,PR,804.0,64.0,1195.0,2035.0,2169.0,2091.0,3347.0,2271.0,2091.0,"2023,2024,2025",A,SUL
22,RS,435.0,10.0,1339.0,1330.0,1957.0,2408.0,3709.0,2375.0,1957.0,"2023,2024,2025",A,SUL
23,SC,311.0,11.0,507.0,1034.0,1913.0,2145.0,2926.0,2112.0,1913.0,"2023,2024,2025",A,SUL
10,MG,231.0,32.0,634.0,785.0,1356.0,1606.0,4399.0,2887.0,1356.0,"2023,2024,2025",A,SUDESTE
6,DF,418.0,41.0,656.0,675.0,1210.0,1301.0,1745.0,1153.0,1210.0,"2023,2024,2025",A,CENTRO_OESTE
4,BA,154.0,35.0,316.0,478.0,970.0,1721.0,2216.0,1795.0,970.0,"2023,2024,2025",A,NORDESTE
5,CE,74.0,46.0,121.0,375.0,1432.0,843.0,1725.0,879.0,843.0,"2023,2024,2025",A,NORDESTE
18,RJ,460.0,192.0,701.0,1011.0,806.0,1842.0,3457.0,2744.0,806.0,"2023,2024,2025",A,SUDESTE
8,GO,233.0,44.0,297.0,453.0,777.0,897.0,1586.0,1094.0,777.0,"2023,2024,2025",A,CENTRO_OESTE


## 9. Validações

Falham com erro (não com aviso): coerência da hierarquia, consistência triângulo × painel, ausência de zeros fabricados em semanas sem arquivo, unicidade da chave do painel, VSR corrigido ≥ VSR confirmado.

In [17]:
CHAVE_PAINEL = ["criterio_uf", "nivel", "unidade", "faixa_etaria", "semana"]


def validar(painel: pd.DataFrame, tri: pd.DataFrame, casos: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    res = []

    def check(nome: str, cond: bool, detalhe: str = ""):
        res.append({"verificacao": nome, "ok": bool(cond), "detalhe": detalhe})

    check("chave do painel única", not painel.duplicated(CHAVE_PAINEL).any())
    valores = list(SERIES_CONTAGEM) + list(LIMITES_VSR_EST)
    for crit in cfg.criterios_uf:
        p = painel.loc[(painel["criterio_uf"] == crit) & painel["coberta"]]
        uf = p.loc[p["nivel"] == "UF"].groupby(["semana", "faixa_etaria"])[valores].sum()
        reg = p.loc[p["nivel"] == "REGIAO"].groupby(["semana", "faixa_etaria"])[valores].sum()
        br = p.loc[p["nivel"] == "BRASIL"].set_index(["semana", "faixa_etaria"])[valores]
        check(f"[{crit}] BR = Σ regiões = Σ UFs",
              np.allclose(uf.sort_index(), br.sort_index()) and np.allclose(reg.sort_index(), br.sort_index()))
        fx = p.loc[p["faixa_etaria"] != FAIXA_TOTAL].groupby(["nivel", "unidade", "semana"])[valores].sum()
        tt = p.loc[p["faixa_etaria"] == FAIXA_TOTAL].set_index(["nivel", "unidade", "semana"])[valores]
        check(f"[{crit}] Σ faixas = total etário", np.allclose(fx.sort_index(), tt.sort_index()))
        for serie in ("vsr", "srag"):
            t = tri.loc[(tri["criterio_uf"] == crit) & (tri["serie"] == serie)].groupby("semana")["casos"].sum()
            b = p.loc[(p["nivel"] == "BRASIL") & (p["faixa_etaria"] == FAIXA_TOTAL)].set_index("semana")[serie]
            t = t.reindex(b.index, fill_value=0)
            check(f"[{crit}] triângulo {serie} soma ao painel", np.allclose(t, b))
    sem_arq = painel.loc[~painel["coberta"], "vsr"]
    check("semanas sem arquivo são NaN", sem_arq.isna().all(), f"{len(sem_arq)} linhas")
    for s in LIMITES_VSR_EST:
        check(f"{s} ≥ vsr", (painel[s].fillna(0) >= painel["vsr"].fillna(0) - 1e-9).all())
    check("positividade em [0,1]", all(painel[c].dropna().between(0, 1).all()
                                           for c in ("positividade_painel", "positividade_lab")))
    chave_temp = ["criterio_uf", "nivel", "unidade", "faixa_etaria", "temporada", "semana_temporada"]
    check("(temporada, semana_temporada) único e em 1..53",
          not painel.duplicated(chave_temp).any() and painel["semana_temporada"].between(1, 53).all())
    check("painel termina na última semana completa",
          painel["semana"].max() + pd.Timedelta(days=6) <= DATA_CORTE.normalize(), str(painel["semana"].max().date()))
    check("casos com semana", casos["semana"].notna().all())
    check("triângulo sem atraso negativo", (tri["atraso"] >= 0).all())
    out = pd.DataFrame(res)
    if not out["ok"].all():
        display(out)
        raise AssertionError("validações falharam: " + ", ".join(out.loc[~out["ok"], "verificacao"]))
    return out


QC_VALIDACAO = validar(PAINEL, TRI, CASOS, CFG)
QC_VALIDACAO

,verificacao,ok,detalhe
0,chave do painel única,True,
1,[residencia] BR = Σ regiões = Σ UFs,True,
2,[residencia] Σ faixas = total etário,True,
3,[residencia] triângulo vsr soma ao painel,True,
4,[residencia] triângulo srag soma ao painel,True,
5,[notificacao] BR = Σ regiões = Σ UFs,True,
6,[notificacao] Σ faixas = total etário,True,
7,[notificacao] triângulo vsr soma ao painel,True,
8,[notificacao] triângulo srag soma ao painel,True,
9,semanas sem arquivo são NaN,True,0 linhas


## 10. Gravação e metadados

In [18]:
DESCRICAO_PAINEL = {
    "criterio_uf": "residencia (principal) | notificacao (sensibilidade)",
    "nivel": "BRASIL | REGIAO | UF", "unidade": "BR, região ou sigla da UF",
    "faixa_etaria": f"00-05m, 06-11m, 12-23m ou {FAIXA_TOTAL} (soma)",
    "semana": "domingo da semana epidemiológica de primeiros sintomas",
    "srag": "SRAG hospitalizada", "vsr": "VSR confirmado (PCR ou antígeno)",
    "vsr_pcr": "VSR por RT-PCR", "vsr_an": "VSR por antígeno",
    "lab_resultado": "com resultado laboratorial (PCR ou antígeno, positivo ou negativo)",
    "testado_painel": "VSR positivo ou POS_PCROUT/POS_AN_OUT ∈ {1,2}; esses campos só são preenchidos com "
                      "agente detectado ⇒ ≈ teste com algum agente detectado (exclui negativos)",
    "sars2": "SARS-CoV-2 por PCR", "vsr_uti": "VSR com UTI", "vsr_obito": "VSR com óbito",
    "vsr_est_inf": "LIMITE INFERIOR do VSR corrigido pela testagem: srag × positividade trailing com shrinkage, "
                   "denominador lab_resultado (inclui negativos de testes só para SARS-CoV-2); coerente; ≥ vsr",
    "vsr_est_sup": "LIMITE SUPERIOR do VSR corrigido pela testagem: mesmo estimador com denominador testado_painel "
                   "(exclui todos os negativos); coerente; ≥ vsr",
    "positividade_painel": "vsr / testado_painel (limite superior da positividade)",
    "positividade_lab": "vsr / lab_resultado (limite inferior da positividade)",
    "frac_testado_painel": "testado_painel / srag",
    "cobertura_arquivos": "fração dos casos da semana vinda de arquivos presentes",
    "coberta": "False ⇒ contagens NaN", "periodo": "pre_pandemia | quebra_npi | pos_pandemia | imunizacao",
    "temporada": "ano da temporada", "semana_temporada": "1..53 dentro da temporada (SE 53 não é achatada)",
    "atraso_ate_corte": "semanas entre a semana e a data de corte (limitado a D)",
    "completude_esperada_vsr": "F_r(atraso_ate_corte) para VSR",
    "semana_incompleta": f"completude_esperada_vsr < {CFG.completude_flag}",
    "*_imput_ref": "observado; em quebra_npi, climatologia de todas as temporadas de referência (EDA)",
    "*_imput_passado": "observado; em quebra_npi, climatologia só de temporadas anteriores (sem vazamento)",
}


def salvar(cfg: Config) -> dict:
    proc, interim = cfg.dir_processed, cfg.dir_interim
    arquivos = {
        interim / "casos_menores2.parquet": CASOS,
        proc / "painel_semanal.parquet": PAINEL[CHAVE_PAINEL + [c for c in PAINEL.columns if c not in CHAVE_PAINEL]]
                                         .sort_values(CHAVE_PAINEL).reset_index(drop=True),
        proc / "triangulo_atraso.parquet": TRI,
        proc / "completude_atraso.parquet": COMP,
        proc / "climatologia.parquet": CLIM,
    }
    for caminho, df in arquivos.items():
        df.to_parquet(caminho, index=False, engine="pyarrow", compression="zstd")
    UNIDADES.to_csv(proc / "hierarquia_unidades.csv", index=False)
    MATRIZ_S.to_csv(proc / "matriz_S.csv")
    TIERS.to_csv(proc / "tiers_uf.csv", index=False)
    QC_FILTROS.to_csv(proc / "qc_filtros.csv", index=False)
    QC_ANO.to_csv(proc / "qc_por_ano.csv")
    QC_COLUNAS.to_csv(proc / "qc_colunas_por_arquivo.csv")
    QC_VALIDACAO.to_csv(proc / "qc_validacao.csv", index=False)

    meta = {
        "gerado_em_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "data_corte": str(DATA_CORTE.date()),
        "semana_final_painel": str(SEMANA_FINAL.date()),
        "serie_alvo": "vsr",
        "vsr_est_limites": {
            "inferior": "vsr_est_inf (denominador lab_resultado = PCR_RESUL ou RES_AN ∈ {1,2}, ou VSR positivo)",
            "superior": "vsr_est_sup (denominador testado_painel)",
            "nota": "limites, não estimativas pontuais: POS_PCROUT, POS_PCRFLU, PCR_FLUASU e PCR_VSR só são "
                    "preenchidos com PCR detectável, logo testado_painel exclui os negativos (superior), e "
                    "lab_resultado inclui negativos de testes só para SARS-CoV-2 (inferior)",
        },
        "hospital_ausente_incluido": cfg.incluir_hospital_ausente,
        "fontes": MANIFESTO,
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(cfg).items()},
        "temporadas_referencia": REFS,
        "faixa_total": FAIXA_TOTAL,
        "chave_painel": CHAVE_PAINEL,
        "qc_atraso": QC_ATRASO,
        "colunas_painel": DESCRICAO_PAINEL,
        "versoes": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
        "arquivos": {str(p.relative_to(cfg.raiz)): {"linhas": len(df), "sha256": sha256(p)}
                     for p, df in arquivos.items()},
    }
    (proc / "metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return meta


META = salvar(CFG)
pd.DataFrame(META["arquivos"]).T

,linhas,sha256
interim\casos_menores2.parquet,585232,d3acaa7c12ab192cca0a18a733de295235e7dc84125ec1...
processed\painel_semanal.parquet,107448,f11f96067e1f0aacc37127155d9467e9201cef9bc70dac...
processed\triangulo_atraso.parquet,321553,3fd8eca07eb128670a0424c3f57d94d175ceddce224e71...
processed\completude_atraso.parquet,324,770b501d775b8eb04931684016bbf2b07ce0de0279b0b0...
processed\climatologia.parquet,447744,bb8950dd55ebb854b55bbc304cf73ec0c31bdfc46b8bbb...


## 11. Uso nos notebooks seguintes

```python
import pandas as pd, json
P = "data/processed"
meta   = json.load(open(f"{P}/metadata.json"))
painel = pd.read_parquet(f"{P}/painel_semanal.parquet",
                         filters=[("criterio_uf", "==", "residencia"), ("faixa_etaria", "==", meta["faixa_total"])])
tri    = pd.read_parquet(f"{P}/triangulo_atraso.parquet")

def versao(tri, origem, serie="vsr", criterio="residencia"):
    # contagem por UF e semana conhecida no fim da semana `origem` (domingo)
    o = pd.Timestamp(origem)
    t = tri[(tri.serie == serie) & (tri.criterio_uf == criterio) & (tri.semana <= o)]
    t = t[t.atraso <= (o - t.semana).dt.days // 7]
    return t.groupby(["uf", "semana"])["casos"].sum()
```

Regras para os modelos: alvo principal `vsr` (sensibilidade: `srag`; `vsr_est_inf`/`vsr_est_sup` só como limites); treinar com `coberta` e, por padrão, sem `quebra_npi` (ou com dummy); usar só a climatologia `tipo == "expanding"` com `temporada_alvo` igual à temporada prevista; avaliar 2026 (`imunizacao`) como teste fora de distribuição; não usar `*_imput_ref` em avaliação fora da amostra.